# Chroma CRUD Operations

This notebook walks through create, read, update, and delete operations with a local Chroma vector store.

In [15]:
import os
import shutil
from pathlib import Path
from uuid import uuid4
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

## 1. Set Up Paths and the Vector Store

In [16]:
# Resolve the project root so the notebook works from either the repo root or the notebooks folder.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

project_root

WindowsPath('c:/Users/Pallavi/Desktop/Advance_RAG_Code/04_vector_stores')

In [17]:
# Load environment variables from the local .env file.
dotenv_path = project_root / ".env"
load_dotenv(dotenv_path=dotenv_path)

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Please add your OPENAI_API_KEY to the .env file before running this notebook.")

print(f"Loaded environment from: {dotenv_path}")

Loaded environment from: c:\Users\Pallavi\Desktop\Advance_RAG_Code\04_vector_stores\.env


In [18]:
# Use a fixed collection name and persistence path so each rerun is predictable.
collection_name = "demo_2"
persist_directory = project_root / "notebooks" / "chroma_langchain_db"

print(f"Collection name: {collection_name}")
print(f"Persist directory: {persist_directory}")

Collection name: demo_2
Persist directory: c:\Users\Pallavi\Desktop\Advance_RAG_Code\04_vector_stores\notebooks\chroma_langchain_db


In [ ]:
# Start fresh so the CRUD flow produces the same result each time.
# Re-running this notebook without restarting the kernel keeps the previous
# Chroma client (and its open chroma.sqlite3 handle) alive via the `vector_store`
# variable, which causes a PermissionError on Windows when we try to delete the
# directory below. Dropping that reference and clearing Chroma's cached clients
# releases the file handle first.
import gc
from chromadb.api.client import SharedSystemClient

if "vector_store" in globals():
    del vector_store

SharedSystemClient.clear_system_cache()
gc.collect()

if persist_directory.exists():
    shutil.rmtree(persist_directory)
    print("Removed the old Chroma directory.")
else:
    print("No previous Chroma directory was found.")

In [20]:
# Create the embedding model and connect it to a persistent Chroma store.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector_store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
    persist_directory=str(persist_directory),
)

print("Vector store is ready.")

Vector store is ready.


## 2. Add Small Helper Functions

In [21]:
def preview_text(text, limit=80):
    """Return a short preview for cleaner notebook output."""
    if len(text) <= limit:
        return text
    return text[:limit] + "..."


def print_documents(title, docs):
    """Print Document objects in a beginner-friendly format."""
    print(title)
    for index, doc in enumerate(docs, start=1):
        print(f"{index}. id={doc.id}")
        print(f"   topic={doc.metadata.get('topic')} | doc_number={doc.metadata.get('doc_number')}")
        print(f"   content={doc.page_content}")
    print()

## 3. Create and Insert Example Documents

In [22]:
# Keep the raw sample data separate from the Document objects so it is easier to read.
document_examples = [
    {
        "topic": "AI",
        "doc_number": 1,
        "text": "Artificial intelligence helps machines perform tasks that usually need human reasoning.",
    },
    {
        "topic": "AI",
        "doc_number": 2,
        "text": "AI systems can analyze patterns in data to support predictions and automation.",
    },
    {
        "topic": "AI",
        "doc_number": 3,
        "text": "Responsible AI development includes fairness, transparency, and safety checks.",
    },
    {
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG combines retrieval with generation so the model can answer using external knowledge.",
    },
    {
        "topic": "RAG",
        "doc_number": 5,
        "text": "A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.",
    },
    {
        "topic": "RAG",
        "doc_number": 6,
        "text": "Vector stores are important in RAG because they make semantic search over embedded documents possible.",
    },
    {
        "topic": "LLM",
        "doc_number": 7,
        "text": "LLMs generate text by predicting likely next tokens from patterns learned during training.",
    },
    {
        "topic": "LLM",
        "doc_number": 8,
        "text": "Prompt design can improve how clearly an LLM follows instructions and returns useful answers.",
    },
    {
        "topic": "Cricket",
        "doc_number": 9,
        "text": "Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.",
    },
    {
        "topic": "Cricket",
        "doc_number": 10,
        "text": "A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.",
    },
]

print(f"Prepared {len(document_examples)} document examples.")

Prepared 10 document examples.


In [23]:
for doc in document_examples:
    print(doc)
    print()

{'topic': 'AI', 'doc_number': 1, 'text': 'Artificial intelligence helps machines perform tasks that usually need human reasoning.'}

{'topic': 'AI', 'doc_number': 2, 'text': 'AI systems can analyze patterns in data to support predictions and automation.'}

{'topic': 'AI', 'doc_number': 3, 'text': 'Responsible AI development includes fairness, transparency, and safety checks.'}

{'topic': 'RAG', 'doc_number': 4, 'text': 'RAG combines retrieval with generation so the model can answer using external knowledge.'}

{'topic': 'RAG', 'doc_number': 5, 'text': 'A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.'}

{'topic': 'RAG', 'doc_number': 6, 'text': 'Vector stores are important in RAG because they make semantic search over embedded documents possible.'}

{'topic': 'LLM', 'doc_number': 7, 'text': 'LLMs generate text by predicting likely next tokens from patterns learned during training.'}

{'topic': 'LLM', 'doc_number': 8, 'text': 'Prompt des

In [24]:
print(uuid4())

696cd8e6-04a9-487d-a971-d1e85b215451


In [25]:
# Convert the sample data into LangChain Document objects.
documents = [
    Document(
        id=str(uuid4()),
        page_content=item["text"],
        metadata={"topic": item["topic"], "doc_number": item["doc_number"]},
    )
    for item in document_examples
]

print_documents("Dummy documents prepared:", documents)

Dummy documents prepared:
1. id=9e2f0134-cef4-4b63-aeef-772e2f0ed0aa
   topic=AI | doc_number=1
   content=Artificial intelligence helps machines perform tasks that usually need human reasoning.
2. id=41517ee8-2444-47b2-9815-7f1ee9d92473
   topic=AI | doc_number=2
   content=AI systems can analyze patterns in data to support predictions and automation.
3. id=311acff4-f7c3-46cb-9899-1fee37bd0589
   topic=AI | doc_number=3
   content=Responsible AI development includes fairness, transparency, and safety checks.
4. id=6edfa624-e908-4397-99d7-348e5fc88635
   topic=RAG | doc_number=4
   content=RAG combines retrieval with generation so the model can answer using external knowledge.
5. id=24b75278-5621-4425-87c6-352a3485fa24
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.
6. id=de1ef426-12df-44fe-96c5-e7099e01af47
   topic=RAG | doc_number=6
   content=Vector stores are important in RAG because they mak

In [26]:
documents[0].id

'9e2f0134-cef4-4b63-aeef-772e2f0ed0aa'

In [27]:
# Insert the documents into Chroma. Chroma creates embeddings during this step.
document_ids = vector_store.add_documents(documents)

print("Inserted document ids:")
for doc_id in document_ids:
    print(doc_id)

print(f"\nTotal inserted documents: {len(document_ids)}")

Inserted document ids:
9e2f0134-cef4-4b63-aeef-772e2f0ed0aa
41517ee8-2444-47b2-9815-7f1ee9d92473
311acff4-f7c3-46cb-9899-1fee37bd0589
6edfa624-e908-4397-99d7-348e5fc88635
24b75278-5621-4425-87c6-352a3485fa24
de1ef426-12df-44fe-96c5-e7099e01af47
14592542-2075-426e-8c1a-7eec4af87d9c
e4cd8d45-f88b-4217-a475-c808856a8d5c
7f280b21-43d6-46b6-a7e0-c0e37c5b9ffa
93ff6b2a-0c27-416b-9f89-24cc7023a9ac

Total inserted documents: 10


## 4. Read the Stored Data Back

In [28]:
# The get() method returns the low-level Chroma record structure.
raw_records = vector_store.get(include=["embeddings", "metadatas", "documents"])
raw_records.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])

In [29]:
raw_records

{'ids': ['9e2f0134-cef4-4b63-aeef-772e2f0ed0aa',
  '41517ee8-2444-47b2-9815-7f1ee9d92473',
  '311acff4-f7c3-46cb-9899-1fee37bd0589',
  '6edfa624-e908-4397-99d7-348e5fc88635',
  '24b75278-5621-4425-87c6-352a3485fa24',
  'de1ef426-12df-44fe-96c5-e7099e01af47',
  '14592542-2075-426e-8c1a-7eec4af87d9c',
  'e4cd8d45-f88b-4217-a475-c808856a8d5c',
  '7f280b21-43d6-46b6-a7e0-c0e37c5b9ffa',
  '93ff6b2a-0c27-416b-9f89-24cc7023a9ac'],
 'embeddings': array([[ 0.00489426,  0.02098083,  0.0165863 , ...,  0.00072622,
         -0.0164032 ,  0.02793884],
        [-0.01451111, -0.00540543,  0.03039551, ..., -0.02856445,
         -0.00189972,  0.04272461],
        [ 0.02272034,  0.01531982,  0.04547119, ...,  0.022995  ,
          0.00803375, -0.01583862],
        ...,
        [ 0.00803375,  0.02314758,  0.02174377, ..., -0.02070618,
         -0.0178833 ,  0.01335144],
        [ 0.00884247,  0.06274414,  0.09973145, ..., -0.01583862,
         -0.01374817,  0.0368042 ],
        [ 0.01802063,  0.08544922, 

In [30]:
print(raw_records["embeddings"][0:2, 0:20].shape)

(2, 20)


In [31]:
print(f"Total records in collection: {len(raw_records['ids'])}")
print("First three ids from get():")
for doc_id in raw_records["ids"][:3]:
    print(doc_id)

Total records in collection: 10
First three ids from get():
9e2f0134-cef4-4b63-aeef-772e2f0ed0aa
41517ee8-2444-47b2-9815-7f1ee9d92473
311acff4-f7c3-46cb-9899-1fee37bd0589


In [32]:
# Pick a few ids so we can read them back in a higher-level format.
selected_ids = document_ids[-3:]
selected_ids

['e4cd8d45-f88b-4217-a475-c808856a8d5c',
 '7f280b21-43d6-46b6-a7e0-c0e37c5b9ffa',
 '93ff6b2a-0c27-416b-9f89-24cc7023a9ac']

In [33]:
# get_by_ids() returns LangChain Document objects instead of the raw Chroma dictionary.
selected_documents = vector_store.get_by_ids(selected_ids)
print_documents("Documents fetched with get_by_ids():", selected_documents)

Documents fetched with get_by_ids():
1. id=e4cd8d45-f88b-4217-a475-c808856a8d5c
   topic=LLM | doc_number=8
   content=Prompt design can improve how clearly an LLM follows instructions and returns useful answers.
2. id=7f280b21-43d6-46b6-a7e0-c0e37c5b9ffa
   topic=Cricket | doc_number=9
   content=Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.
3. id=93ff6b2a-0c27-416b-9f89-24cc7023a9ac
   topic=Cricket | doc_number=10
   content=A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.



In [34]:
print(selected_documents)

[Document(id='e4cd8d45-f88b-4217-a475-c808856a8d5c', metadata={'topic': 'LLM', 'doc_number': 8}, page_content='Prompt design can improve how clearly an LLM follows instructions and returns useful answers.'), Document(id='7f280b21-43d6-46b6-a7e0-c0e37c5b9ffa', metadata={'topic': 'Cricket', 'doc_number': 9}, page_content='Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.'), Document(id='93ff6b2a-0c27-416b-9f89-24cc7023a9ac', metadata={'doc_number': 10, 'topic': 'Cricket'}, page_content='A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.')]


## 5. Run a Similarity Search

In [35]:
query = "How does RAG help an LLM answer questions using outside knowledge?"
query

'How does RAG help an LLM answer questions using outside knowledge?'

In [36]:
search_results = vector_store.similarity_search(query, k=3)
print(f"Query: {query}\n")
print_documents("Similarity search results:", search_results)

Query: How does RAG help an LLM answer questions using outside knowledge?

Similarity search results:
1. id=6edfa624-e908-4397-99d7-348e5fc88635
   topic=RAG | doc_number=4
   content=RAG combines retrieval with generation so the model can answer using external knowledge.
2. id=e4cd8d45-f88b-4217-a475-c808856a8d5c
   topic=LLM | doc_number=8
   content=Prompt design can improve how clearly an LLM follows instructions and returns useful answers.
3. id=24b75278-5621-4425-87c6-352a3485fa24
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.



In [37]:
search_results

[Document(id='6edfa624-e908-4397-99d7-348e5fc88635', metadata={'doc_number': 4, 'topic': 'RAG'}, page_content='RAG combines retrieval with generation so the model can answer using external knowledge.'),
 Document(id='e4cd8d45-f88b-4217-a475-c808856a8d5c', metadata={'topic': 'LLM', 'doc_number': 8}, page_content='Prompt design can improve how clearly an LLM follows instructions and returns useful answers.'),
 Document(id='24b75278-5621-4425-87c6-352a3485fa24', metadata={'doc_number': 5, 'topic': 'RAG'}, page_content='A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.')]

In [38]:
vector_store.similarity_search_with_score(query=query, k=4)

[(Document(id='6edfa624-e908-4397-99d7-348e5fc88635', metadata={'doc_number': 4, 'topic': 'RAG'}, page_content='RAG combines retrieval with generation so the model can answer using external knowledge.'),
  0.8067883253097534),
 (Document(id='e4cd8d45-f88b-4217-a475-c808856a8d5c', metadata={'doc_number': 8, 'topic': 'LLM'}, page_content='Prompt design can improve how clearly an LLM follows instructions and returns useful answers.'),
  0.9388180375099182),
 (Document(id='24b75278-5621-4425-87c6-352a3485fa24', metadata={'doc_number': 5, 'topic': 'RAG'}, page_content='A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.'),
  1.0751897096633911),
 (Document(id='14592542-2075-426e-8c1a-7eec4af87d9c', metadata={'topic': 'LLM', 'doc_number': 7}, page_content='LLMs generate text by predicting likely next tokens from patterns learned during training.'),
  1.1304855346679688)]

## 6. Update Existing Documents

In [39]:
# We will update one RAG document and one LLM document.
ids_to_update = [document_ids[3], document_ids[7]]
ids_to_update

['6edfa624-e908-4397-99d7-348e5fc88635',
 'e4cd8d45-f88b-4217-a475-c808856a8d5c']

In [40]:
# Keep the replacement text separate so the update step stays easy to follow.
updated_examples = [
    {
        "id": ids_to_update[0],
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG improves answer quality by retrieving relevant context before the language model generates a response.",
    },
    {
        "id": ids_to_update[1],
        "topic": "LLM",
        "doc_number": 8,
        "text": "Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.",
    },
]

updated_documents = [
    Document(
        id=item["id"],
        page_content=item["text"],
        metadata={"topic": item["topic"], "doc_number": item["doc_number"]},
    )
    for item in updated_examples
]

print_documents("Updated document content:", updated_documents)

Updated document content:
1. id=6edfa624-e908-4397-99d7-348e5fc88635
   topic=RAG | doc_number=4
   content=RAG improves answer quality by retrieving relevant context before the language model generates a response.
2. id=e4cd8d45-f88b-4217-a475-c808856a8d5c
   topic=LLM | doc_number=8
   content=Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.



In [41]:
print([doc.page_content for doc in documents if doc.id in ids_to_update])

['RAG combines retrieval with generation so the model can answer using external knowledge.', 'Prompt design can improve how clearly an LLM follows instructions and returns useful answers.']


In [42]:
vector_store.update_documents(ids=ids_to_update, documents=updated_documents)

print("Updated these ids:")
for doc_id in ids_to_update:
    print(doc_id)

Updated these ids:
6edfa624-e908-4397-99d7-348e5fc88635
e4cd8d45-f88b-4217-a475-c808856a8d5c


In [43]:
# Read the updated records back from Chroma to confirm the new values were stored.
updated_raw_records = vector_store.get(ids=ids_to_update)

print("Raw records returned by get(ids=ids_to_update):")
for doc_id, document_text, metadata in zip(
    updated_raw_records["ids"],
    updated_raw_records["documents"],
    updated_raw_records["metadatas"],
):
    print(f"id={doc_id}")
    print(f"metadata={metadata}")
    print(f"content={preview_text(document_text)}")
    print()

Raw records returned by get(ids=ids_to_update):
id=6edfa624-e908-4397-99d7-348e5fc88635
metadata={'doc_number': 4, 'topic': 'RAG'}
content=RAG improves answer quality by retrieving relevant context before the language m...

id=e4cd8d45-f88b-4217-a475-c808856a8d5c
metadata={'topic': 'LLM', 'doc_number': 8}
content=Well-written prompts help an LLM stay focused, follow instructions, and produce ...



In [44]:
updated_query = "How can retrieved context improve an LLM response in RAG?"
updated_query

'How can retrieved context improve an LLM response in RAG?'

In [45]:
updated_search_results = vector_store.similarity_search(updated_query, k=2)
print(f"Updated query: {updated_query}\n")
print_documents("Similarity search after update:", updated_search_results)

Updated query: How can retrieved context improve an LLM response in RAG?

Similarity search after update:
1. id=6edfa624-e908-4397-99d7-348e5fc88635
   topic=RAG | doc_number=4
   content=RAG improves answer quality by retrieving relevant context before the language model generates a response.
2. id=24b75278-5621-4425-87c6-352a3485fa24
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.



## 7. Delete Documents

In [46]:
# Delete the two cricket examples so the final collection is smaller.
ids_to_delete = [document_ids[8], document_ids[9]]
ids_to_delete

['7f280b21-43d6-46b6-a7e0-c0e37c5b9ffa',
 '93ff6b2a-0c27-416b-9f89-24cc7023a9ac']

In [47]:
vector_store.delete(ids=ids_to_delete)

print("Deleted these ids:")
for doc_id in ids_to_delete:
    print(doc_id)

Deleted these ids:
7f280b21-43d6-46b6-a7e0-c0e37c5b9ffa
93ff6b2a-0c27-416b-9f89-24cc7023a9ac


In [48]:
remaining_records = vector_store.get()
remaining_ids = remaining_records["ids"]

print(f"Remaining document count: {len(remaining_ids)}")
print("Remaining ids:")
for doc_id in remaining_ids:
    print(doc_id)

print("\nDeleted ids still present?")
for doc_id in ids_to_delete:
    print(f"{doc_id}: {doc_id in remaining_ids}")

Remaining document count: 8
Remaining ids:
9e2f0134-cef4-4b63-aeef-772e2f0ed0aa
41517ee8-2444-47b2-9815-7f1ee9d92473
311acff4-f7c3-46cb-9899-1fee37bd0589
6edfa624-e908-4397-99d7-348e5fc88635
24b75278-5621-4425-87c6-352a3485fa24
de1ef426-12df-44fe-96c5-e7099e01af47
14592542-2075-426e-8c1a-7eec4af87d9c
e4cd8d45-f88b-4217-a475-c808856a8d5c

Deleted ids still present?
7f280b21-43d6-46b6-a7e0-c0e37c5b9ffa: False
93ff6b2a-0c27-416b-9f89-24cc7023a9ac: False


In [49]:
print([doc.metadata["topic"] for doc in documents if doc.id in remaining_ids])

['AI', 'AI', 'AI', 'RAG', 'RAG', 'RAG', 'LLM', 'LLM']
